# Figure panel 7

In [1]:
options(warn=-1)

In [2]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(
        
            # Seurat 
            library(Seurat), 

            # Data 
            library(tidyverse), 
            library(data.table), 


            # Plotting 
            library(ggplot2), 
            library(patchwork), 
            library(ComplexHeatmap), 
            library(circlize), 
            library(viridis), 
            library(ggplotify), 
            library(ggrepel), 
            library(cowplot), 

            # Pyhton compatibility
            library(reticulate)

        )
    )
)

In [3]:
# Configure reticulate 
# use_condaenv(condaenv="p.3.10.16-FD20200109SPLENO", conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
# py_config()

In [4]:
random_seed <- 42
set.seed(random_seed)

In [5]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20200109SPLENO")

In [6]:
# Source
source("bin/so_pl.R")
source("bin/tradeseq_pp.R")
source("bin/tradeseq_pl.R")

In [7]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=4)) # From project global source()

# Load reference data 

In [8]:
tf <- read.table("/research/peer/fdeckert/reference/animaltfdb/Mus_musculus_TF.txt")[[2]]

lr <- CellChat::CellChatDB.mouse[[1]] %>% dplyr::select(pathway_name, ligand, receptor.symbol, receptor.family, annotation) %>% dplyr::filter(annotation %in% c("Cell-Cell Contact", "Secreted Signaling", "ECM-Receptor")) %>% 
    separate_rows(receptor.symbol, sep=", ") %>% dplyr::distinct() %>%
    separate_rows(ligand, sep=", ") %>% dplyr::distinct() %>% 
    dplyr::rename(pathway=pathway_name, receptor=receptor.symbol, family=receptor.family)

Registered S3 method overwritten by 'ggnetwork':
  method         from  
  fortify.igraph ggtree



# Load data

## scRNAseq data

In [9]:
so <- readRDS("data/scRNAseq/object/pp_1.rds")
so <- NormalizeData(so)

Normalizing layer: counts



## Erythroid lineage fitgam and make contrasts

In [10]:
tradeseq_res <- readRDS("result/lineage/tradeseq_res_2.rds")
fitgam <- tradeseq_res[["fitgam"]]
ptpg <- tradeseq_res[["ptpg"]]

In [11]:
contrast_1 <- c("IFNAR_fl_Baseline_D0", "IFNAR_fl_CpG_D1")
ptpg_res_1 <- ptpg[ptpg[["contrast"]]==paste0(contrast_1, collapse=":"), ]

contrast_2 <- c("IFNAR_fl_Baseline_D0", "IFNAR_fl_CpG_D3")
ptpg_res_2 <- ptpg[ptpg[["contrast"]]==paste0(contrast_2, collapse=":"), ]

contrast_3 <- c("IFNAR_fl_CpG_D1", "IFNAR_fl_CpG_D3")
ptpg_res_3 <- ptpg[ptpg[["contrast"]]==paste0(contrast_3, collapse=":"), ]



contrast_4 <- c("IFNAR_fl_LysM_cre_Baseline_D0", "IFNAR_fl_LysM_cre_CpG_D1")
ptpg_res_4 <- ptpg[ptpg[["contrast"]]==paste0(contrast_4, collapse=":"), ]

contrast_5 <- c("IFNAR_fl_LysM_cre_Baseline_D0", "IFNAR_fl_LysM_cre_CpG_D3")
ptpg_res_5 <- ptpg[ptpg[["contrast"]]==paste0(contrast_5, collapse=":"), ]

contrast_6 <- c("IFNAR_fl_LysM_cre_CpG_D1", "IFNAR_fl_LysM_cre_CpG_D3")
ptpg_res_6 <- ptpg[ptpg[["contrast"]]==paste0(contrast_6, collapse=":"), ]



contrast_7 <- c("IFNAR_fl_Baseline_D0", "IFNAR_fl_LysM_cre_Baseline_D0")
ptpg_res_7 <- ptpg[ptpg[["contrast"]]==paste0(contrast_7, collapse=":"), ]

contrast_8 <- c("IFNAR_fl_CpG_D1", "IFNAR_fl_LysM_cre_CpG_D1")
ptpg_res_8 <- ptpg[ptpg[["cozntrast"]]==paste0(contrast_8, collapse=":"), ]

contrast_9 <- c("IFNAR_fl_CpG_D3", "IFNAR_fl_LysM_cre_CpG_D3")
ptpg_res_9 <- ptpg[ptpg[["contrast"]]==paste0(contrast_9, collapse=":"), ]

# Erythroid progenitor marker 

All BFU-E potential in the cKit+ CD71low/Ter119low population resided in the CD150+ (Slamf1) fraction. Exclusion of mature erythroid precursor cells in spleen was facilitated by inclusion of CD24a as a negative selection marker. Of the surface markers tested (CD9, CD11a, CD34, CD48, CD63, and CD79b), only CD9 could further fractionate the BFU-E-containing CD150+ population in stressed spleen. To further discriminate putative multi-potent stress-progenitors from lineage restricted stress-BFU-E we also included Sca1, which in steady-state BM separates Sca1+ (Ly6a) hematopoietic stem cells (HSC) and Sca1– myelo-erythroid progenitors.

## Module score 

In [ ]:
so <- readRDS("data/scRNAseq/object/pp.rds")
so <- NormalizeData(so)

In [ ]:
fate_simplex_fwd <- read.csv("result/cellrank/fate_simplex_fwd_p_vbc.csv", row.names=1) 
fate_simplex_fwd <- fate_simplex_fwd[rownames(fate_simplex_fwd) %in% colnames(so), ]
colnames(fate_simplex_fwd) <- c("CR_1", "CR_2")

In [ ]:
so[["CR"]] <- SeuratObject::CreateDimReducObject(embeddings=as.matrix(fate_simplex_fwd), key="CR_", assay="RNA")

In [ ]:
so <- so[, so$celltype_low %in% c("GMP", "NeuP", "BasoP", "MastP", "MegP", "MEP", "Proerythroblast")]

In [ ]:
features <- list(

    msP_1=c("Hlf+", "Esam+"), 
    msP_2=c("Kit+", "Flt3+"), 
    msP_3=c("Cd24a+", "Tfrc+"), 
    msP_4=c("Slamf1+"),
    msP_5=c("Fcgr2b+", "Fcgr3+"), 
    msP_6=c("Ms4a2+"),  
    msP_7=c("Elane+", "Prtn3+"), 
    msP_8=c("Gzmb+"), 
    msP_9=c("Prss34+"), 
    msP_10=c("Ms4a2+", "Gzmb+", "Prss34+", "Fcgr2b+", "Fcgr3+", "Cd34+"), 
    msP_11=c("Cd9+"), 
    msP_12=c("Ly6a+"), 
    msP_13=c("Gata1+", "Klf1+", "Gata2-")


)

In [ ]:
so <- UCell::AddModuleScore_UCell(so, features=features, assay="RNA", slot="counts", name="", maxRank=2500)

In [ ]:
pt_size <- 0.5
order <- TRUE

max_set <- 1
min_cutoff <- 0
max_cutoff <- 1

In [ ]:
p_l <- list(

    dplot(so, group_by="celltype_low", group_color=color$celltype_low, reduction="CR", pt_size=pt_size, legend_position="none", size_select=4), 
    fplot(so, features="msP_1", assay="RNA", slot="data", reduction="CR", pt_size=pt_size, order=order, max_set=max_set, min_cutoff=min_cutoff, max_cutoff=max_cutoff, size_select=4, color_end=0.95), 
    fplot(so, features="msP_2", assay="RNA", slot="data", reduction="CR", pt_size=pt_size, order=order, max_set=max_set, min_cutoff=min_cutoff, max_cutoff=max_cutoff, size_select=4, color_end=0.95), 
    fplot(so, features="msP_3", assay="RNA", slot="data", reduction="CR", pt_size=pt_size, order=order, max_set=max_set, min_cutoff=min_cutoff, max_cutoff=max_cutoff, size_select=4, color_end=0.95), 
    fplot(so, features="msP_4", assay="RNA", slot="data", reduction="CR", pt_size=pt_size, order=order, max_set=max_set, min_cutoff=min_cutoff, max_cutoff=max_cutoff, size_select=4, color_end=0.95), 
    fplot(so, features="msP_5", assay="RNA", slot="data", reduction="CR", pt_size=pt_size, order=order, max_set=max_set, min_cutoff=min_cutoff, max_cutoff=max_cutoff, size_select=4, color_end=0.95), 
    fplot(so, features="msP_6", assay="RNA", slot="data", reduction="CR", pt_size=pt_size, order=order, max_set=max_set, min_cutoff=min_cutoff, max_cutoff=max_cutoff, size_select=4, color_end=0.95), 
    fplot(so, features="msP_7", assay="RNA", slot="data", reduction="CR", pt_size=pt_size, order=order, max_set=max_set, min_cutoff=min_cutoff, max_cutoff=max_cutoff, size_select=4, color_end=0.95), 
    fplot(so, features="msP_8", assay="RNA", slot="data", reduction="CR", pt_size=pt_size, order=order, max_set=max_set, min_cutoff=min_cutoff, max_cutoff=max_cutoff, size_select=4, color_end=0.95), 
    fplot(so, features="msP_9", assay="RNA", slot="data", reduction="CR", pt_size=pt_size, order=order, max_set=max_set, min_cutoff=min_cutoff, max_cutoff=max_cutoff, size_select=4, color_end=0.95), 
    fplot(so, features="msP_10", assay="RNA", slot="data", reduction="CR", pt_size=pt_size, order=order, max_set=max_set, min_cutoff=min_cutoff, max_cutoff=max_cutoff, size_select=4, color_end=0.95), 
    fplot(so, features="msP_11", assay="RNA", slot="data", reduction="CR", pt_size=pt_size, order=order, max_set=max_set, min_cutoff=min_cutoff, max_cutoff=max_cutoff, size_select=4, color_end=0.95), 
    fplot(so, features="msP_12", assay="RNA", slot="data", reduction="CR", pt_size=pt_size, order=order, max_set=max_set, min_cutoff=min_cutoff, max_cutoff=max_cutoff, size_select=4, color_end=0.95)
    
)

In [ ]:
p_l <- lapply(p_l, function(p) egg::set_panel_size(p, width=unit(3, "cm"), height=unit(3, "cm")))
p_l <- do.call(gridExtra::arrangeGrob, c(p_l, ncol=10, nrow=ceiling(length(p_l)/10)))

In [ ]:
pdf("result/figures/1_scRNAseq/panel_7/cr_p_marker_genes.pdf", width=10*2, height=2*ceiling(length(p_l)/10))

grid::grid.draw(p_l)

dev.off()